First, you'll need to fill in your OpenAI API key and an organization and project ID (created within your OpenAI API account).

Then, work through all the sections down to the next marked text section. These configure the classes and functions to enable easy interaction with the OpenAI API.


In [ ]:
#[1] Access parameters
#The key is obtained from the AI/API Keys website.
#Important: Load credit in the Open AI account.
OPENAI_API_KEY_DO_NOT_SHARE = ''
#Using this field only if the key is associated to an organization in OpenAI. If they key is not associated to any specific organization the fielf can be empty.
OPENAI_ORGANIZATION = ''
 #This field is usefel if you manage multiple projects in the same organization. It is not necessary if you only manage the API with a standard account.
OPENAI_PROJECT_ID = ''

In [ ]:
#[2] Instalation of libraries to interact with the OpenAI API and manage text tokenization in Google Colab.
!pip install openai
!pip install tiktoken
!pip install python-docx



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 7.6 MB/s eta 0:00:00


In [ ]:
#[3] Importing several libraries to manage files, times, tokanization, JSON, API errors and retry strategies.
import pickle
import time
import datetime
import tiktoken
import json
import traceback
import numpy as np
from tqdm.notebook import tqdm
from jsonschema import validate
from openai import OpenAI, RateLimitError, APITimeoutError, InternalServerError, Timeout
from tenacity import retry, stop_after_attempt, wait_incrementing, retry_if_exception_type, after_log, before_sleep_log
import logging

In [ ]:
#[4] Rate and token limits are based on the user/organization's OpenAI account usage level.
rate_limit_per_minute = 1e3  # Max  1000 request per minute
token_limit_per_minute = 1e6  # Max 1,000,000 tokens per minute

#Calculate the minimal pause between solicitudes to avoid overpass the requests per minute limit (If the rate limit per minute = 1000, pause between each request will be 60 / 1000 = 0.06 seconds.
request_delay_seconds = 60.0 / rate_limit_per_minute  # Not change
request_timeout_seconds = 120   # Maximum time to wait before discarding a request
request_max_retries = 1         # Maximum number of retries on failure
tpm_wait_polling_seconds = 10   # Time to wait before checking if the token limit has been reset

# Global logger for static classes
logger = logging.getLogger()
logger.setLevel(logging.INFO)

In [ ]:
#[5] Defining a ChatGPT class to interact with the OpenAI API, incorporating error handling, usage limits, and response validation.
class ChatGPT:
    def __init__(self, model, #Initializes a client to make requests to OpenAI.
                 halt_on_error=True,
                 is_verbose=True,
                 timeout=request_timeout_seconds,
                 max_retries=request_max_retries,
                 request_delay_seconds=request_delay_seconds,
                 token_limit_per_minute=token_limit_per_minute,
                 tpm_wait_polling_seconds=tpm_wait_polling_seconds,
                 logger=logger,
                 api_key=OPENAI_API_KEY_DO_NOT_SHARE,
                 organization=OPENAI_ORGANIZATION,
                 project=OPENAI_PROJECT_ID):
        self.model = model
        self.halt_on_error = halt_on_error
        self.is_verbose = is_verbose
        self.tpm_wait_polling_seconds = tpm_wait_polling_seconds
        self.request_delay_seconds = request_delay_seconds
        self.token_limit_per_minute = token_limit_per_minute
        self.response_history = []
        self.message_history = {}
        self.token_count_history = []
        self.logger = logger

        supported_models = ['gpt-4o-mini', 'gpt-4o-mini-2024-07-18', 'gpt-4o-2024-08-06']
        if self.model not in supported_models:
            raise Exception(f'{model} is not supported. Model must be one of {supported_models}')

        self.client = OpenAI(api_key = api_key,
                             organization = organization,
                             project = project,
                             timeout=timeout,
                             max_retries=max_retries)


    def get_running_cost_num_prompt_completion_tokens(self): #Estimates the cost of tokens used in API requests.
        """
        This function computes the total cost (estimated) of all
        messages sent by the instance of ChatGPT called from
        Returns: total_running_cost, total_num_prompt_tokens, total_num_response_tokens
        """
        if self.model == 'gpt-4o-2024-08-06':
            prompt_cost = 0.0025 / 1000
            response_cost = 0.01 / 1000
        elif self.model == 'gpt-4o-mini':
            prompt_cost = 0.00015 / 1000
            response_cost = 0.0006 / 1000
        elif self.model == 'gpt-4o-mini-2024-07-18':
            prompt_cost = 0.00015 / 1000
            response_cost = 0.0006 / 1000
        else:
            prompt_cost = np.nan
            response_cost = np.nan

        n_prompt_tokens = np.sum([x.usage.prompt_tokens for x in self.response_history])
        n_completion_tokens = np.sum([x.usage.completion_tokens for x in self.response_history])
        return ((n_prompt_tokens * prompt_cost) + (n_completion_tokens * response_cost),
                n_prompt_tokens,
                n_completion_tokens)


    # Retry failing requests starting with 10 second wait,
    # Increasing wait time by 10 seconds each retry, up to a max window of 120s (or 5 times)
    # The goal is to try to avoid hitting backoff,
    # We treat this as a last resort because of its runtime cost
    @retry(wait=wait_incrementing(start=10, increment=10, max=120),
           stop=stop_after_attempt(5),
           retry=retry_if_exception_type((RateLimitError, APITimeoutError, InternalServerError, Timeout)),
           before_sleep=before_sleep_log(logger, logging.INFO),
           after=after_log(logger, logging.INFO))
    def completion_with_backoff(self, client, **kwargs):  #Use retry to retry requests in case of errors (e.g. usage limits).
        return client.chat.completions.create(**kwargs)


    def check_internal_TPM_tracker(self, n_message_tokens): #Control how many tokens have been sent in the last minute. If the limit is exceeded, wait before sending more requests.
        """
        Checks internal TPM count to see if a message with length = n_message_tokens
        can be sent. If not, it waits (sleeps - blocking) until the message delivery
        meets into TPM limit
        """
        now = datetime.datetime.now()
        one_minute_ago = now + datetime.timedelta(seconds=-60)
        self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
        n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        # Fixed delay waiting if TPM exceeded over past minute
        # This is cpu polling, so it doesnt cost money or much compute
        while n_tokens_past_minute > self.token_limit_per_minute:
            if self.is_verbose: self.logger.info(f'Internal TPM limit exceeded, waiting for {self.tpm_wait_polling_seconds} seconds...')
            time.sleep(self.tpm_wait_polling_seconds)
            now = datetime.datetime.now()
            one_minute_ago = now + datetime.timedelta(seconds=-60)
            self.token_count_history = [x for x in self.token_count_history if x[1] > one_minute_ago]
            n_tokens_past_minute = np.sum([x[0] for x in self.token_count_history]) + n_message_tokens
        now = datetime.datetime.now()
        self.token_count_history.append((n_message_tokens, now))


    def send_message(self, system_role, message, json_schema):#Check that the request's json_schema contains certain required fields. Control the request and token limits per minute. Send the message to the OpenAI API with a conversation history. Handle errors and validate the API response.
        """
        This is the primary function used to send messages to GPT and get responses
        Steps are:
          - check that json schema meets our basic requirements
          - handle RPM and TPM limits as best as we can
            (when openai rejects requests for exceeding limits its much slower)
          - build and send the message using openai ChatCompletion api
          - perform basic validation on GPT's response
        This function either returns a ChatCompletion response object or None (if failure occurred)
        Errors are propogated using raised Exceptions
        """
        # Check json_schema fullfil minimun requeriments
        if 'refusal' not in json_schema['schema']['properties']:
            return self.bad_response_output(f'Output JSON schema must contain the field "refusal" under "properties". Missing in `json_schema` passed as input.')
        if 'reason_for_refusal' not in json_schema['schema']['properties']:
            return self.bad_response_output(f'Output JSON schema must contain the field "reason_for_refusal" under "properties". Missing in `json_schema` passed as input.')

        # Sleep based on RPM limit (lazy logic, avoids keeping a running count of actual requests per minute)
        time.sleep(self.request_delay_seconds)

        # Check the TPM limit (not lazy, uses continuous token count per minute)
        encoding = tiktoken.encoding_for_model(self.model)
        n_message_tokens = len(encoding.encode(system_role)) + len(encoding.encode(message))
        if n_message_tokens > self.token_limit_per_minute:
            return self.bad_response_output(f'Unable to send message as it exceeds TPM. Number of tokens in message = {n_message_tokens}')
        self.check_internal_TPM_tracker(n_message_tokens)

        # Build and send messages through OpenAI-API
        message_id = len(self.message_history.keys())
        self.message_history[message_id] = [{"role": "system", "content": system_role}]
        self.message_history[message_id].append({"role": "user", "content": message})
        try:
            response = self.completion_with_backoff(self.client,
                                                    model=self.model,
                                                    messages=self.message_history[message_id],
                                                    temperature=0,
                                                    stream=False,
                                                    response_format={"type": "json_schema",
                                                                     "json_schema": json_schema},
                                                    seed=0, logprobs=True)
        except Exception as e:
            if self.halt_on_error:
                raise
            else:
                if self.is_verbose:
                    str_e = str(e)
                    self.logger.info(f'An exception occurred: {str_e}')
                    self.logger.info(traceback.format_exc())
                return None

        self.response_history.append(response)

        # Response validation, check if GPT's finish_reason is valid
        finish_reason = response.choices[0].finish_reason
        if finish_reason != 'stop':
            return self.bad_response_output(f'Invalid GPT finish reason: {finish_reason}')

        # Response validation, check if json contents are valid
        if not self.json_response_validation(response, json_schema):
            return self.bad_response_output(f'GPT response failed validation')
        return (response, message_id)


    def bad_response_output(self, error): #Logs errors and decides whether to stop execution or continue.
        # General function to inform the user when an error occurs.
        if self.halt_on_error:
            raise Exception(error)
        else:
            if self.is_verbose:
                self.logger.info(f'Error - {error}')
        return None


    def json_response_validation(self, response, json_schema): #Checks that the OpenAI response is valid JSON and conforms to the expected schema. Rejects responses that the model refuses to respond to.
          # Basic request validation: All GPT responses are checked with this logic.
          # To validate additional data or responses, the user must define additional wrapper functions.
          # for the send_message function call.
        try:
            # Checks that it is a valid JSON string.
            response_json = json.loads(response.choices[0].message.content)
            # Checks that the JSON content matches the schema.
            validate(instance=response_json,
                     schema=json_schema['schema'])
            # Checks if the response is valid.
            if bool(response_json['refusal']):
                refusal_reason = response_json['reason_for_refusal']
                raise Exception(f'GPT refused to respond. Reason: "{refusal_reason}"')
            else:
                return True
        except Exception as e:
            if self.is_verbose:
                str_e = str(e)
                self.logger.info(f'Failed to validate response - {str_e}')
            return False

# Stop Here
Now there are a few additional things you'll need to change. By running the next cell, you'll be able to load a CSV document with the data. Right now, it's set up to expect a file like a Google Doc and takes the ResponseID and Translated Story columns. If your CSV file has a different format, you'll need to change a few things in the next cells.

In [ ]:
#[6] Load a xlsx file
from google.colab import files
uploaded = files.upload()

Saving Goals - Dataframe.xlsx to Goals - Dataframe.xlsx


In [ ]:
#[7] Read a xlsx file in the pandas dataframe. File name must be adjusted.
import pandas as pd
import io

#pd.read_excel(...) Read a xlsx file. Use pd.read_csv if the file has a csv format.
df = pd.read_excel(io.BytesIO(uploaded['Goals - Dataframe.xlsx']))
print(df)

INFO:numexpr.utils:NumExpr defaulting to 2 threads.


             ResponseId                                               Goal
0     R_3K15tYj7HHWmT1I  Basic description: 'Learn English'. Additional...
1     R_3K15tYj7HHWmT1I  Basic description: 'Earn 10 million'. Addition...
2     R_7NRbO83E104AAr7  Basic description: 'Learn English'. Additional...
3     R_7NRbO83E104AAr7  Basic description: 'Study software engineering...
4     R_57y9Sdj22ll2ao6  Basic description: 'CANCEL MORTGAGE LOAN'. Add...
...                 ...                                                ...
3180         3013859641  Basic description: 'Provide a good education f...
3181         3022451673  Basic description: 'a better job position'. Ad...
3182         3142451537  Basic description: 'Lose more than 10 kg of we...
3183         3208370390  Basic description: 'Launch my app and have my ...
3184         3212753553  Basic description: 'My goal in 4 months is to ...

[3185 rows x 2 columns]


In [ ]:
#[8]Extract specific columns from a DataFrame (df) and converts them to a more manageable format for processing with GPT.
##In this case, only the ResponseID and the Goal are taken, which will be used to traverse the GPT calls
transcript_data = df[["ResponseId","Goal"]]
transcript_data = transcript_data.values
transcript_data

array([['R_3K15tYj7HHWmT1I',
        "Basic description: 'Learn English'. Additional information: 'Travel to another country and be able to interact without problems'."],
       ['R_3K15tYj7HHWmT1I',
        "Basic description: 'Earn 10 million'. Additional information: 'Live more freely without worrying so much about money'."],
       ['R_7NRbO83E104AAr7',
        "Basic description: 'Learn English'. Additional information: 'Speak fluent English to get a good job and better salaries'."],
       ...,
       ['3142451537',
        "Basic description: 'Lose more than 10 kg of weight'. Additional information: 'For health, aesthetics, and a bet with my girlfriend for an iPad, since mid-January I have wanted to achieve it and I am fully motivated to accomplish this goal'."],
       ['3208370390',
        "Basic description: 'Launch my app and have my tech startup.' Additional information: 'It is important to me because I can do what I love, develop software, and work remotely. I want to ach

Now run the following cells to configure the system role, message, and JSON schema for responses.

In [ ]:
#[9] Define the system role
QA_system_role = ("You are an AI assistant helping an expert in social psychology who is studying the relationship between socioeconomic status and the pursuit of personal goals.")

In [ ]:
#[10] Function that generates a message with embedded transcript data
# This includes the full message text, and the transcript will be appended at the end before being sent to GPT
#generate_QA_user_message(...) Creates a message that integrates the transcript into a suitable format to be sent to GPT
def generate_QA_user_message(transcript):
    transcript_json = {"transcript": transcript.strip()} #transcript.strip(...): Removes whitespace at the beginning and end of the transcript. "transcript_json = " creates a dictionary with the key "transcript", storing the transcript text.
    transcript_json_str = json.dumps(transcript_json) #json.dumps(...): Converts the dictionary into a JSON string (str), ensuring it is sent in a structured format.

#Builds the message that will be sent to GPT.
#The JSON transcript is appended at the end of the message so GPT can access the information.
    message = """
Context
We are studying the relationship between socioeconomic status and the achievement of personal goals. We define personal goals as the outcomes a person aims to achieve, guided by their needs, interests, values, and aspirations.
 Previous studies have shown that people with fewer economic resources face greater difficulties in making progress toward their goals. Our objective is to explore which variables might explain why this pattern of difficulties happens.
 To address this, we conducted a longitudinal study in which participants from different socioeconomic backgrounds described their personal goals. Some time later, we evaluated the progress they had made toward these goals.
 One possible explanation for the differences in progress may be related to the types of goals each group pursues. Therefore, we need to classify the reported goals to identify the kinds of goals people in our study are pursuing.
 Each participant provided:
•	 A basic description of their goal (what they want to achieve).
•	Additional information (why the goal is important, how long they have been pursuing it, and how they feel about it).
Role
You are an AI assistant helping an expert in social psychology who is studying the relationship between socioeconomic status and the pursuit of personal goals.
 Task description
Review all the goals provided by the participants and build a taxonomy that classifies the personal goals according to the life domains to which the goals belong.
 The taxonomy must meet the following criteria:
•	 Clarity and distinction: Create clear and well-differentiated categories. Avoid redundancies or categories that are too similar.
•	 Grouping of equivalents: If you identify goals that are essentially the same (for example, "Improve physical health" and "Take care of my physical condition"), group them under a single category.
•	 Relevance: Do not include irrelevant categories. If you find meaningless or nonsensical responses, assign them to a category called "Not applicable" instead of creating a new category.
•	 Conceptual independence: Do not rely on existing taxonomies as references. Build the classification solely based on the information provided in the participants' goals.
 Output format:
Present each category of the taxonomy as a bullet point, and for each category include:
•	 A clear and descriptive name.
•	A brief definition of the category.
•	Two concrete examples taken directly from the provided goals.

    """ + transcript_json_str
    return message

In [ ]:
#[11] Inicializating GPT
chat = ChatGPT(model='gpt-4o-2024-08-06')

In [ ]:
#[12] Define the wrapper function to send messages to GPT and handle errors
def QA_messaging_wrapper(chat, system_role, message, json_schema):
    # Initialize variables
    message_history_id = -1  # Initializing to -1 (will be updated if the request is successful).
    taxonomia = None  # Will store the taxonomy generated by GPT.

    try:
        # Send the message to GPT and get the response
        response, message_history_id = chat.send_message(system_role=system_role,
                                                         message=message,
                                                         json_schema=json_schema)

        # Check that the response is not None
        if response is not None:
            # Convert the text response to a JSON dictionary
            response_json = json.loads(response.choices[0].message.content)

            # Check if GPT refused to respond
            if bool(response_json.get('refusal', False)):
                refusal_reason = response_json.get('reason_for_refusal', 'No reason provided')
                chat.logger.warning(f'GPT refused to respond. Reason: "{refusal_reason}"')
            else:
                # Extract the taxonomy from the JSON response
                taxonomia = response_json.get('taxonomia', [])
        else:
            chat.logger.warning("GPT returned None response.")

    except Exception as e:
        # Handle any exceptions that occur during the process
        chat.logger.error(f'Messaging wrapper failure - {str(e)}')
        taxonomia = None  # In case of error, set taxonomy to None

    # Get the cost of processing (tokens used in the prompt and in the response)
    cost, n_prompt_tokens, n_completion_tokens = chat.get_running_cost_num_prompt_completion_tokens()

    # Return the taxonomy and the processing cost
    return taxonomia, cost

In [ ]:
# [13] Sending messages to GPT and generating the Word file

# Define the JSON schema we expect as a response from GPT
json_schema = {
    "name": "SelfControlStrategiesTaxonomy",  # Schema name
    "schema": {
        "type": "object",
        "properties": {
            "taxonomia": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "nombre_categoria": {"type": "string"},
                        "definicion": {"type": "string"},
                        "ejemplos": {
                            "type": "array",
                            "items": {"type": "string"},
                            "minItems": 1,  # Changed from 2 to 1
                            "maxItems": 2   # Keep a maximum of 2
                        }
                    },
                    "required": ["nombre_categoria", "definicion", "ejemplos"]
                }
            },
            "refusal": {"type": "boolean"},
            "reason_for_refusal": {"type": "string"}
        },
        "required": ["taxonomia", "refusal", "reason_for_refusal"]
    }
}

# List to store GPT responses
complete_taxonomy = []

# Loop through each goal and send it to GPT
for i, (external_ref, goal) in enumerate(transcript_data):
    user_message = generate_QA_user_message(goal)

    # Use the wrapper function to send the message and handle errors
    taxonomia, cost = QA_messaging_wrapper(chat, QA_system_role, user_message, json_schema)

    if taxonomia:
        complete_taxonomy.extend(taxonomia)  # Add the taxonomy to the list
        print(f"Processed goal {i+1}/{len(transcript_data)}")
    else:
        print(f"Error processing goal {i+1}. Continuing with the next...")

# Here you can print or manipulate complete_taxonomy as needed
print("Generated categories:")
for categoria in complete_taxonomy:
    print(f"Category: {categoria['nombre_categoria']}")
    print(f"Definition: {categoria['definicion']}")
    print("Examples:")
    for ejemplo in categoria['ejemplos']:
        print(f"- {ejemplo}")
    print()  # Space between categories

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Procesada situación 1/3185


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Procesada situación 2/3185


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Procesada situación 25/3185


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Procesada situación 26/3185


KeyboardInterrupt: 

In [ ]:
# [14] Second script: Category refinement and Word file generation

# Let's assume complete_taxonomy is the list generated by the first script
# complete_taxonomy = [...]  # This list has already been generated by the first script

# Define the new prompt for refining the categories
refinement_prompt = """
The following is a list of categories of personal goals that people pursue. I want you to review all the categories, their definitions, and examples, and formulate a new version of the refined taxonomy. If there are two or more categories referring to the same type of goals, group them. As a result, create a Word document with the final refined categories, their definitions, and two examples for each.
"""

# Convert the category list into a format that GPT can process
categories_for_prompt = []
for category in complete_taxonomy:
    category_str = f"Category: {category['nombre_categoria']}\n"
    category_str += f"Definition: {category['definicion']}\n"
    category_str += "Examples:\n"
    for example in category['ejemplos']:
        category_str += f"- {example}\n"
    categories_for_prompt.append(category_str)

# Join all categories into a single text to send to GPT
categories_text = "\n".join(categories_for_prompt)

# Send the prompt to GPT to refine the categories
response_refinement, message_id = chat.send_message(
    system_role="You are an AI assistant that refines taxonomies of self-control strategies.",
    message=f"{refinement_prompt}\n\n{categories_text}",
    json_schema=json_schema  # We use the same JSON schema for the response
)

# Check if GPT returned a valid response
if response_refinement:
    try:
        # Convert the text response to a JSON dictionary
        response_json = json.loads(response_refinement.choices[0].message.content)

        # Check if GPT refused to respond
        if bool(response_json.get('refusal', False)):
            refusal_reason = response_json.get('reason_for_refusal', 'No reason provided')
            print(f'GPT refused to respond. Reason: "{refusal_reason}"')
            refined_taxonomy = []
        else:
            # Extract the refined taxonomy from the JSON response
            refined_taxonomy = response_json.get('taxonomia', [])
            print("Refined taxonomy generated by GPT.")
    except json.JSONDecodeError as e:
        print(f"Error parsing GPT response: {e}")
        refined_taxonomy = []
    except Exception as e:
        print(f"Unexpected error processing GPT response: {e}")
        refined_taxonomy = []
else:
    print("Error refining the taxonomy with GPT.")
    refined_taxonomy = []

# Generate the Word file with the refined taxonomy
from docx import Document

doc = Document()
doc.add_heading('Self-Control Strategies Taxonomy (Refined)', 0)

for category in refined_taxonomy:
    doc.add_heading(category['nombre_categoria'], level=1)
    doc.add_paragraph(f"Definition: {category['definicion']}")
    doc.add_paragraph("Examples:")
    for example in category['ejemplos']:
        doc.add_paragraph(f"- {example}")
    doc.add_paragraph()

# Save the Word file
doc.save('Refined_SelfControl_Strategies_Taxonomy.docx')
print("Word file generated and saved as 'Refined_SelfControl_Strategies_Taxonomy.docx'")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Taxonomía depurada generada por GPT.
Archivo de Word generado y guardado como 'Taxonomia_Estrategias_Autocontrol_Depurada.docx'


In [ ]:
# [15] from google.colab import files

# Download the generated file
files.download('Refined_SelfControl_Strategies_Taxonomy.docx')

FileNotFoundError: Cannot find file: Taxonomia_Estrategias_Autocontrol_Depurada.docx